<a href="https://colab.research.google.com/github/228w1a1281/Segmentation-with-SAM-and-Automatic-Shapefile-Generation/blob/main/Mini1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -O sam_vit_h_4b8939.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

--2025-07-28 14:27:29--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.227.219.33, 13.227.219.59, 13.227.219.10, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.227.219.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2564550879 (2.4G) [binary/octet-stream]
Saving to: ‘sam_vit_h_4b8939.pth’

sam_vit_h_4b8939.pt 100%[===================>]   2.39G   174MB/s    in 21s     

2025-07-28 14:27:50 (117 MB/s) - ‘sam_vit_h_4b8939.pth’ saved [2564550879/2564550879]



In [ ]:
!pip install torch torchvision
!pip install git+https://github.com/facebookresearch/segment-anything.git
!pip install opencv-python-headless rasterio shapely geopandas scikit-image

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-99emrff3
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-99emrff3
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 71.6 MB/s eta 0:00:00


In [ ]:
!pip install affine
!pip install opencv-python-headless
!pip install geopandas
!pip install scikit-image


In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install segment-anything opencv-python matplotlib numpy geopandas shapely pillow gdal pyproj fiona scikit-image --quiet

from google.colab import files as colab_files
uploaded = colab_files.upload()

import os
import zipfile
import torch
import numpy as np
import cv2
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import geopandas as gpd
from shapely.geometry import Polygon
from osgeo import gdal
from PIL import Image
from skimage import feature, filters
from scipy import ndimage

def load_sam_model():
    checkpoint_path = "sam_vit_h_4b8939.pth"
    sam = sam_model_registry["vit_h"](checkpoint=checkpoint_path)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    sam.to(device=device)

    mask_generator = SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=32,
        pred_iou_thresh=0.86,
        stability_score_thresh=0.92,
        crop_n_layers=1,
        crop_n_points_downscale_factor=2,
        min_mask_region_area=100,
    )
    return mask_generator

def extract_features(mask, image):
    if len(image.shape) == 2:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 1:
        image_rgb = cv2.cvtColor(image[:,:,0], cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 2:
        image_rgb = np.stack([image[:,:,0]]*3, axis=-1)
    elif image.shape[2] >= 3:
        image_rgb = image[:,:,:3]
    else:
        raise ValueError(f"Unsupported image shape: {image.shape}")

    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

    mask = mask.astype(bool)
    masked_rgb = image_rgb[mask]
    masked_hsv = hsv[mask]
    masked_gray = gray[mask]

    mean_color = np.mean(masked_rgb, axis=0) if masked_rgb.size > 0 else np.zeros(3)
    hsv_mean = np.mean(masked_hsv, axis=0) if masked_hsv.size > 0 else np.zeros(3)

    if masked_gray.size > 0:
        if len(masked_gray.shape) > 1:
            masked_gray = masked_gray.reshape(-1)
        normalized_gray = masked_gray.astype(float) / 255.0
        edges = feature.canny(gray.astype(float)/255.0, mask=mask)
        edge_density = np.sum(edges) / np.sum(mask) if np.sum(mask) > 0 else 0
    else:
        edge_density = 0

    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        contour = max(contours, key=cv2.contourArea)
        perimeter = cv2.arcLength(contour, True)
        area = cv2.contourArea(contour)
        circularity = 4 * np.pi * area / (perimeter**2) if perimeter > 0 else 0
    else:
        circularity = 0

    return {
        'mean_color': mean_color,
        'hsv_mean': hsv_mean,
        'edge_density': edge_density,
        'circularity': circularity,
        'area': np.sum(mask)
    }

def classify_object(features):
    area = features['area']
    color = features['mean_color']
    hsv = features['hsv_mean']
    edge_density = features['edge_density']
    circularity = features['circularity']

    if (hsv[0] > 90 and hsv[0] < 140) and (hsv[1] > 30) and edge_density < 0.1:
        return "Water Body", "blue"
    elif (0.4 < circularity < 0.8) and (area > 5000) and (color[2] < 200):
        return "Building", "red"
    elif (hsv[0] > 30 and hsv[0] < 90) and (hsv[1] > 40) and edge_density > 0.15:
        return "Tree", "green"
    elif (area > 3000) and (circularity < 0.3) and (hsv[1] < 40):
        return "Road", "black"
    elif (hsv[0] > 30 and hsv[0] < 90) and (2000 < area < 8000):
        return "Garden", "lightgreen"
    elif (area < 1500) and (circularity > 0.7):
        return "Statue", "gray"
    elif area > 10000:
        return "Large Structure", "orange"
    else:
        return "Unknown", "purple"

def clean_mask(mask):
    mask = ndimage.binary_fill_holes(mask)
    mask = filters.gaussian(mask, sigma=1) > 0.5
    return mask

def mask_to_polygons(mask):
    mask = clean_mask(mask)
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for contour in contours:
        if len(contour) >= 3:
            epsilon = 0.005 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True)
            polygon = Polygon([tuple(pt[0]) for pt in approx])
            if polygon.is_valid and not polygon.is_empty:
                polygons.append(polygon)
    return polygons

def pixel_to_geo_transform(dataset, x, y):
    gt = dataset.GetGeoTransform()
    x_geo = gt[0] + x * gt[1] + y * gt[2]
    y_geo = gt[3] + x * gt[4] + y * gt[5]
    return x_geo, y_geo

def calculate_shapiness_index(polygon):
    area = polygon.area
    perimeter = polygon.length

    if perimeter == 0:
        return 0

    circularity = (4 * np.pi * area) / (perimeter ** 2)
    shapiness_index = circularity

    return circularity

def segment_and_export(tiff_filename):
    dataset = gdal.Open(tiff_filename)
    if dataset is None:
        raise ValueError("Could not open the TIFF file")

    bands = []
    for i in range(dataset.RasterCount):
        band = dataset.GetRasterBand(i+1).ReadAsArray()
        bands.append(band)

    if len(bands) >= 3:
        image = np.dstack(bands[:3])
    elif len(bands) == 2:
        image = np.dstack([bands[0], bands[1], bands[0]])
    else:
        image = np.dstack([bands[0]]*3)

    if image.dtype != np.uint8:
        image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    mask_generator = load_sam_model()
    masks = mask_generator.generate(image)

    if not masks:
        print("No masks generated - try adjusting parameters")
        return

    results = []
    for m in masks:
        mask = m["segmentation"]
        features = extract_features(mask, image)
        label, color = classify_object(features)
        polygons = mask_to_polygons(mask)

        for poly in polygons:
            results.append({
                "geometry": poly,
                "label": label,
                "color": color,
                "area": features['area'],
            })

    geoms, labels, colors, areas = [], [], [], []
    for res in results:
        coords = [(int(pt[0]), int(pt[1])) for pt in res["geometry"].exterior.coords]
        geo_coords = [pixel_to_geo_transform(dataset, x, y) for (x, y) in coords]
        polygon = Polygon(geo_coords)
        if polygon.is_valid:
            geoms.append(polygon)
            labels.append(res["label"])
            colors.append(res["color"])
            areas.append(res["area"])

    if not geoms:
        print("No valid geometries created")
        return

    gdf = gpd.GeoDataFrame({
        "class": labels,
        "color": colors,
        "area": areas,
        "geometry": geoms
    }, crs="EPSG:4326")

    output_folder = "shapefile_output"
    os.makedirs(output_folder, exist_ok=True)
    shapefile_path = os.path.join(output_folder, "segmented_shapes.shp")
    gdf.to_file(shapefile_path)

    zip_path = "segmented_shapefile.zip"
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for root, _, files in os.walk(output_folder):
            for file in files:
                zipf.write(os.path.join(root, file), arcname=file)

    print(f"Shapefile successfully created with {len(gdf)} features!")

if uploaded:
    tiff_filename = next(iter(uploaded))
    segment_and_export(tiff_filename)
    colab_files.download("segmented_shapefile.zip")
else:
    print("No file uploaded")

Saving c1_modified.tif to c1_modified.tif
Shapefile successfully created with 407 features!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>